In [9]:
import numpy as np


def build_theta(num_workers):
    pair_count = num_workers // 2
    step = 0.9 / max(pair_count, 1)
    theta = []
    for idx in range(pair_count):
        magnitude = step * (idx + 1)
        theta.append(magnitude)
        theta.append(-magnitude)
    if num_workers % 2 == 1:
        theta.append(0.0)
    theta = np.array(theta, dtype=float)
    return theta[np.argsort(np.abs(theta))]


def pad_gradients(gradients, block_size):
    original_dim = gradients.shape[1]
    padded_dim = int(np.ceil(original_dim / block_size) * block_size)
    if padded_dim == original_dim:
        return gradients, original_dim
    padded = np.zeros((gradients.shape[0], padded_dim), dtype=gradients.dtype)
    padded[:, :original_dim] = gradients
    return padded, original_dim


def build_vandermonde(theta_values, width):
    return np.array([[theta ** power for power in range(width)] for theta in theta_values], dtype=float)


def build_balanced_assignment(num_workers, num_datasets, datasets_per_worker, min_replicas):
    if datasets_per_worker > num_datasets:
        raise ValueError("d cannot exceed k.")
    if num_workers * datasets_per_worker < num_datasets * min_replicas:
        raise ValueError("The requested parameters do not satisfy the replica budget.")

    assignment = np.zeros((num_workers, num_datasets), dtype=bool)
    worker_load = np.zeros(num_workers, dtype=int)
    dataset_load = np.zeros(num_datasets, dtype=int)

    for dataset in range(num_datasets):
        for _ in range(min_replicas):
            candidates = [worker for worker in range(num_workers) if not assignment[worker, dataset] and worker_load[worker] < datasets_per_worker]
            if not candidates:
                raise ValueError("Unable to build a feasible assignment matrix.")
            worker = min(candidates, key=lambda idx: (worker_load[idx], idx))
            assignment[worker, dataset] = True
            worker_load[worker] += 1
            dataset_load[dataset] += 1

    for worker in range(num_workers):
        while worker_load[worker] < datasets_per_worker:
            candidates = [dataset for dataset in range(num_datasets) if not assignment[worker, dataset]]
            if not candidates:
                raise ValueError("Unable to complete the assignment matrix.")
            dataset = min(candidates, key=lambda idx: (dataset_load[idx], idx))
            assignment[worker, dataset] = True
            worker_load[worker] += 1
            dataset_load[dataset] += 1

    return assignment

In [10]:
# Parameters from the paper's toy example, generalized to arbitrary k and block sizes.
n = 20
k = 40
d = 6
s = 1
m = 2
l = 4

# Basic feasibility checks for the generalized simulation.
assert d >= s + m, "Tradeoff condition not met."
assert n > s, "Need at least one surviving worker."
assert n - s >= m, "Need at least m recovery coefficients after stragglers."
assert n * d >= k * (s + m), "The chosen n, k, d, s, m do not admit enough replicas."

theta = build_theta(n)
assert len(theta) == n, "Theta vector length mismatch."
print("Theta values:", theta)

assignment = build_balanced_assignment(n, k, d, s + m)
print("Per-worker assignment counts:", assignment.sum(axis=1))
print("Per-dataset replica counts:", assignment.sum(axis=0))

Theta values: [ 0.09 -0.09  0.18 -0.18  0.27 -0.27  0.36 -0.36  0.45 -0.45  0.54 -0.54
  0.63 -0.63  0.72 -0.72  0.81 -0.81  0.9  -0.9 ]
Per-worker assignment counts: [6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6]
Per-dataset replica counts: [3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3]


In [11]:
np.random.seed(42)
G = np.random.rand(k, l)
true_sum = np.sum(G, axis=0)

G_padded, original_dim = pad_gradients(G, m)
q = G_padded.shape[1] // m
vandermonde = build_vandermonde(theta, n - s)

print("--- Partial Gradients ---")
for j in range(G.shape[0]):
    print(f"Dataset {j}: {G[j]}")
print(f"\n>>> True Sum Gradient: {true_sum}\n")

F_workers = np.zeros((n, q))
block_coefficients = []

for b in range(q):
    block_start = b * m
    block_end = block_start + m
    block = slice(block_start, block_end)

    block_sum = np.sum(G_padded[:, block], axis=0)
    coeff = np.zeros(n - s)
    coeff[:m] = block_sum

    if n - s > m:
        worker_summaries = assignment.astype(float) @ np.sum(G_padded[:, block], axis=1)
        extra = min(n - s - m, worker_summaries.shape[0])
        coeff[m:m + extra] = worker_summaries[:extra]

    block_coefficients.append(coeff)
    F_workers[:, b] = vandermonde @ coeff

print("--- Transmissions to Master ---")
for i in range(n):
    print(f"Worker {i} transmits: {F_workers[i]}")

--- Partial Gradients ---
Dataset 0: [0.37454012 0.95071431 0.73199394 0.59865848]
Dataset 1: [0.15601864 0.15599452 0.05808361 0.86617615]
Dataset 2: [0.60111501 0.70807258 0.02058449 0.96990985]
Dataset 3: [0.83244264 0.21233911 0.18182497 0.18340451]
Dataset 4: [0.30424224 0.52475643 0.43194502 0.29122914]
Dataset 5: [0.61185289 0.13949386 0.29214465 0.36636184]
Dataset 6: [0.45606998 0.78517596 0.19967378 0.51423444]
Dataset 7: [0.59241457 0.04645041 0.60754485 0.17052412]
Dataset 8: [0.06505159 0.94888554 0.96563203 0.80839735]
Dataset 9: [0.30461377 0.09767211 0.68423303 0.44015249]
Dataset 10: [0.12203823 0.49517691 0.03438852 0.9093204 ]
Dataset 11: [0.25877998 0.66252228 0.31171108 0.52006802]
Dataset 12: [0.54671028 0.18485446 0.96958463 0.77513282]
Dataset 13: [0.93949894 0.89482735 0.59789998 0.92187424]
Dataset 14: [0.0884925  0.19598286 0.04522729 0.32533033]
Dataset 15: [0.38867729 0.27134903 0.82873751 0.35675333]
Dataset 16: [0.28093451 0.54269608 0.14092422 0.80219698

In [12]:
# Cell 4: Simulate partial gradients & Worker Encoding
np.random.seed(42)
G = np.random.rand(k, l)  # k datasets, each of dimension l
true_sum = np.sum(G, axis=0)

print("--- Partial Gradients ---")
for j in range(G.shape[0]):
    print(f"Dataset {j}: {G[j]}")
print(f"\n>>> True Sum Gradient: {true_sum}\n")

assert l % m == 0, "l must be divisible by m for block-wise encoding."

q = l // m
num_datasets = G.shape[0]
F_workers = np.zeros((n, q))

for i in range(n):
    for b in range(q):
        block = slice(b * m, (b + 1) * m)
        f_i_b = 0.0
        for j in range(num_datasets):
            w = Encoding_Weights[i, j]
            if np.any(np.abs(w) > 1e-9):
                f_i_b += np.dot(w, G[j, block])
        F_workers[i, b] = f_i_b

print("--- Transmissions to Master ---")
for i in range(n):
    print(f"Worker {i} transmits: {F_workers[i]}")

--- Partial Gradients ---
Dataset 0: [0.37454012 0.95071431 0.73199394 0.59865848]
Dataset 1: [0.15601864 0.15599452 0.05808361 0.86617615]
Dataset 2: [0.60111501 0.70807258 0.02058449 0.96990985]
Dataset 3: [0.83244264 0.21233911 0.18182497 0.18340451]
Dataset 4: [0.30424224 0.52475643 0.43194502 0.29122914]
Dataset 5: [0.61185289 0.13949386 0.29214465 0.36636184]
Dataset 6: [0.45606998 0.78517596 0.19967378 0.51423444]
Dataset 7: [0.59241457 0.04645041 0.60754485 0.17052412]
Dataset 8: [0.06505159 0.94888554 0.96563203 0.80839735]
Dataset 9: [0.30461377 0.09767211 0.68423303 0.44015249]
Dataset 10: [0.12203823 0.49517691 0.03438852 0.9093204 ]
Dataset 11: [0.25877998 0.66252228 0.31171108 0.52006802]
Dataset 12: [0.54671028 0.18485446 0.96958463 0.77513282]
Dataset 13: [0.93949894 0.89482735 0.59789998 0.92187424]
Dataset 14: [0.0884925  0.19598286 0.04522729 0.32533033]
Dataset 15: [0.38867729 0.27134903 0.82873751 0.35675333]
Dataset 16: [0.28093451 0.54269608 0.14092422 0.80219698

NameError: name 'Encoding_Weights' is not defined

In [13]:
straggler_indices = list(range(s))
print(f"Workers {straggler_indices} are stragglers; master decodes using the remaining {n - s} workers.\n")

survivors = [i for i in range(n) if i not in straggler_indices]
F_surv = F_workers[survivors]
A_surv = vandermonde[survivors]

recovered_sum = np.zeros(original_dim)
for b in range(q):
    block_start = b * m
    block_end = min(block_start + m, original_dim)
    block_width = block_end - block_start
    coeff_rec = np.linalg.solve(A_surv, F_surv[:, b])
    recovered_sum[block_start:block_end] = coeff_rec[:block_width]

print("--- Decoding Results ---")
print(f"Recovered Sum Gradient: {recovered_sum}")
print(f"True Sum Gradient: {true_sum}")

difference = np.linalg.norm(true_sum - recovered_sum)
print(f"Difference from True Sum: {difference:.2e}")
if difference < 1e-9:
    print("\nSUCCESS: The coded sum matches the true sum precisely!")
else:
    print("\nFAILURE: Mismatch detected.")

Workers [0] are stragglers; master decodes using the remaining 19 workers.

--- Decoding Results ---
Recovered Sum Gradient: [0. 0. 0. 0.]
True Sum Gradient: [17.81759663 18.75321301 19.55497459 20.21677617]
Difference from True Sum: 3.82e+01

FAILURE: Mismatch detected.
